# exp40 — HTR-VT (CNN + transformer + CTC, from scratch) on exp35's data mix

**Question:** in the augmentation study (htr-metrics-transfer, HHD only) our HTR-VT reached ~6% test CER from scratch, better than the
pretrained TrOCR-CTC (12.45). Does it also hold its own with exp35's full training data, against exp35-large?

**Same as exp35** (`cyttic/exp35-trocr-large-en-ctc-seqhead-allreal-synpunct`): all real lines (HHD train+test minus 10% validation
writers, SCE with density correction, Pinkas) + **20% synthetic digits/punctuation** per epoch (fresh draws, 1,000 held out), Mishkefet
charset and augmentation (strength 0.7, ramped over 3 epochs), checkpoint chosen on HHD validation (greedy CER), final scores with greedy
and beam 12 + Mishkefet char LM.

**Different:** the model = our HTR-VT (htr-metrics-transfer `train_htrvt.ipynb`): residual CNN (64 px line → feature sequence, one CTC
step per 4 px), span masking (40%, spans of 4), 6-layer pre-norm transformer (dim 384, 6 heads), linear CTC head, **random init**.
Input = one continuous line (no 384×384 tiling), squeezed to at most 2,304 px (= exp35's 6 × 384). AdamW lr 5e-4, weight decay 0.05,
warm-up 5% + cosine decay to 1%, batch 32, **100 epochs**, validation every 2 epochs.

**A5000:** `./run.sh exp40/exp40_htrvt_allreal_synpunct_colab.ipynb` inside tmux, ~4–5 h. Output → public HF repo
`cyttic/exp40-htrvt-allreal-synpunct`. The ivrit.ai benchmark is scored afterwards, locally.

### 1. Setup

In [ ]:
!pip install -q -U "transformers==5.15.0" datasets jiwer opencv-python-headless safetensors

In [ ]:
import os, io, re, json, math, time, glob, random, pickle, types
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, jiwer
from PIL import Image
from datasets import load_dataset, concatenate_datasets
from huggingface_hub import snapshot_download, HfApi, login, whoami
try:
    from google.colab import userdata; login(userdata.get("HF_TOKEN"))
except Exception:
    from huggingface_hub import get_token   # outside Colab (e.g. the A5000 server): the token saved by `hf auth login`
    if get_token() is None:
        from huggingface_hub import notebook_login; notebook_login()
print("HF user:", whoami()["name"])
device = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8   # Ampere+ (A5000/L4): bf16
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
print(device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| amp", AMP_DTYPE)

### 2. Config

In [ ]:
DATA_ID  = "cyttic/trocr-hebrew-matan"           # HHD: train 4,740 / test 1,455
MISH_ID  = "itayinbar/Mishkefet-v1"              # CODE + charset + char LM + augmentation only (no weights)
OUT_REPO = "cyttic/exp40-htrvt-allreal-synpunct"   # public
SCE_ID   = "cyttic/trocr-hebrew-human"           # SCE human set: 280 writers, 127 texts
PINKAS_ID        = "cyttic/pinkas-hebrew-lines"  # mirror of Zenodo 3569694 (CC-BY-4.0), line crops
PINKAS_LOCAL_DIR = "/mnt/ssd2/cyttic/datasets/pinkas_lines"   # used instead of HF when it exists
DENSITY_TARGET   = 22.9     # HHD train median px/char at 64 px height; SCE is compressed to it (as exp35)
HHF_ID        = "cyttic/heb-synth-punct-digits-hhf"                   # 20k synthetic lines
HHF_LOCAL_DIR = "/mnt/ssd2/cyttic/datasets/heb_synth_punct_digits_hhf"  # used instead of HF when it exists
SYN_HOLDOUT   = 1000        # synthetic lines never trained on: final digit/punctuation check
SYN_FRAC      = 0.20        # share of each epoch: synthetic (real ~80%)
PUSH     = True

SEED            = 42
VAL_WRITER_FRAC = 0.10      # 10% of HHD writers -> validation (= exp35's split)
ROW_H, MAX_W    = 64, 6 * 384   # line height; wider lines are squeezed to 2304 px (exp35's container width), not cut
BATCH           = 32
LR, WARMUP, WD  = 5e-4, 0.05, 0.05
EPOCHS, EVAL_EVERY = 100, 2
MODEL_CFG = {"cnn_width": 256, "dim": 384, "depth": 6, "heads": 6, "mlp_ratio": 4, "dropout": 0.1, "mask_ratio": 0.4, "mask_span": 4}
AUGMENT, AUG_MAX, AUG_RAMP = True, 0.7, 3   # Mishkefet's augmentation, strength ramped over 3 epochs (as exp35)
NUM_WORKERS     = min(8, os.cpu_count())
SMOKE           = False     # True = tiny run to test the pipeline end to end

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT = "/content/exp40" if os.path.isdir("/content") else os.path.abspath("exp40_run")
os.makedirs(OUT, exist_ok=True); print("OUT", OUT)

### 3. Mishkefet helpers (charset, CTC decoders, char LM, augmentation) — safe loading

In [ ]:
MISH = snapshot_download(MISH_ID, allow_patterns=["hebocr/*", "hebocr/**", "charset.json", "hebrew_char6.pkl"])
import sys; sys.path.insert(0, MISH)

class _NoGlobals(pickle.Unpickler):                  # the LM is a plain pickle: allow data only
    def find_class(self, module, name):
        raise pickle.UnpicklingError(f"blocked global {module}.{name}")
_safe = types.SimpleNamespace(load=lambda fh: _NoGlobals(fh).load(), dump=pickle.dump, HIGHEST_PROTOCOL=pickle.HIGHEST_PROTOCOL)
import hebocr.lm as _lm; _lm.pickle = _safe
from hebocr.charset import Charset, BLANK
from hebocr.decode import greedy_decode, beam_decode
from hebocr.lm import CharNGramLM
from hebocr.data.transforms import Augment

charset = Charset.load(os.path.join(MISH, "charset.json"))
char_lm = CharNGramLM.load(os.path.join(MISH, "hebrew_char6.pkl"))
print("charset:", charset.n_classes, "classes (blank =", BLANK, ")")

### 4. Data (= exp35) and line formatting (continuous line, no tiling)

In [ ]:
STRIDE = 4   # the CNN reduces width 4x -> one CTC step per 4 px
def fmt(img):
    """line image -> gray -> 64 px high -> mirror (RTL -> LTR) -> squeezed to <= MAX_W -> (1, 64, W) in [-1, 1] + CTC length"""
    g = img.convert("L"); w, h = g.size
    nw = min(MAX_W, max(STRIDE, round(w * ROW_H / max(h, 1))))
    a = np.asarray(g.resize((nw, ROW_H), Image.LANCZOS))[:, ::-1]
    t = torch.from_numpy(np.ascontiguousarray(a)).float().div(255).sub(0.5).div(0.5).unsqueeze(0)
    return t, max(1, nw // STRIDE)

COLS = ["image", "text", "writer", "src"]
hhd = load_dataset(DATA_ID)
hhd_all = concatenate_datasets([hhd["train"], hhd["test"]])
hhd_all = hhd_all.map(lambda w: {"writer": "hhd:" + w, "src": "hhd"}, input_columns=["writer"]).select_columns(COLS)
sce = load_dataset(SCE_ID)
sce_all = concatenate_datasets([sce["train"], sce["test"]])
sce_all = sce_all.map(lambda d: {"writer": "sce:" + d, "src": "sce"}, input_columns=["source_doc"]).select_columns(COLS)
if os.path.isdir(PINKAS_LOCAL_DIR):
    pinkas = load_dataset("parquet", data_files={"train": sorted(glob.glob(os.path.join(PINKAS_LOCAL_DIR, "*.parquet")))}, split="train")
else:
    pinkas = load_dataset(PINKAS_ID, split="train")
pinkas = pinkas.map(lambda w: {"src": "pinkas"}, input_columns=["writer"]).select_columns(COLS)

wr = sorted(set(hhd_all["writer"])); random.Random(SEED).shuffle(wr)          # = exp35's validation writers
val_w = set(wr[:max(1, int(len(wr) * VAL_WRITER_FRAC))])
hw = hhd_all["writer"]
val = hhd_all.select([i for i, w in enumerate(hw) if w in val_w])
train = concatenate_datasets([hhd_all.select([i for i, w in enumerate(hw) if w not in val_w]), sce_all, pinkas]).select_columns(COLS)
if SMOKE:
    train = concatenate_datasets([train.filter(lambda s_: s_ == src, input_columns=["src"]).select(range(24)) for src in ("hhd", "sce", "pinkas")])
    val = val.select(range(32)); EPOCHS, EVAL_EVERY = 2, 1

if os.path.isdir(HHF_LOCAL_DIR):
    hhf = load_dataset("parquet", data_files={"train": sorted(glob.glob(os.path.join(HHF_LOCAL_DIR, "*.parquet")))}, split="train")
else:
    hhf = load_dataset(HHF_ID, split="train")
hhf = hhf.shuffle(seed=SEED)
syn_val = hhf.select(range(SYN_HOLDOUT))
syn = hhf.select(range(SYN_HOLDOUT, len(hhf)))
syn = syn.map(lambda st: {"writer": "synth:" + st, "src": "synth"}, input_columns=["style"]).select_columns(COLS)
if SMOKE: syn, syn_val = syn.select(range(60)), syn_val.select(range(24))

def train_set(ep):
    """the lines one epoch trains on: all real + SYN_FRAC synthetic (fresh random draw per epoch, = exp35 phase B)"""
    total = len(train) / (1 - SYN_FRAC)
    ps = random.Random(SEED + ep).sample(range(len(syn)), min(len(syn), int(total * SYN_FRAC)))
    return concatenate_datasets([train, syn.select(ps)])

from collections import Counter
src_lines = Counter(train["src"])
print("train lines by source:", dict(src_lines), "| epoch size:", len(train_set(1)))
print(f"train {len(train)} lines, {len(set(train['writer']))} writers | val {len(val)} lines ({len(val_w)} HHD writers)")
print("label coverage (train):", round(np.mean([charset.coverage(t) for t in train["text"]]), 4))

augment = Augment(strength=AUG_MAX)
def collate(batch, train_mode=False):
    xs, labels = [], []
    for ex in batch:
        img = ex["image"]
        if ex.get("src") == "sce" and ex["text"].strip():          # SCE is 1.6x sparser than HHD (as exp35)
            dens = img.width * 64 / max(img.height, 1) / len(ex["text"])
            f = float(np.clip(DENSITY_TARGET / dens, 0.4, 8.0))
            if abs(f - 1) > 0.02:
                img = img.resize((max(16, int(round(img.width * f))), img.height), Image.BICUBIC)
        if train_mode and AUGMENT:
            img = Image.fromarray(augment.augment_gray(img))
        xs.append(fmt(img)); labels.append(charset.encode(ex["text"]))
    W = int(math.ceil(max(x[0].shape[-1] for x in xs) / STRIDE) * STRIDE)
    px = torch.ones(len(xs), 1, ROW_H, W)                         # pad with white (+1)
    for i, (t, _) in enumerate(xs): px[i, :, :, :t.shape[-1]] = t
    return px, torch.tensor([x[1] for x in xs]), labels, [ex["text"] for ex in batch]

def loader(split, train_mode):
    return torch.utils.data.DataLoader(split, batch_size=BATCH, shuffle=train_mode, num_workers=NUM_WORKERS,
                                       collate_fn=lambda b: collate(b, train_mode), drop_last=False)

### 5. Model: HTR-VT (ours), random init

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, cin, cout, pool):
        super().__init__()
        self.c1, self.b1 = nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout)
        self.c2, self.b2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout)
        self.sc = nn.Identity() if cin == cout else nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), nn.BatchNorm2d(cout))
        self.pool = nn.MaxPool2d(pool) if pool != (1, 1) else nn.Identity()
    def forward(self, x):
        y = self.b2(self.c2(F.relu(self.b1(self.c1(x)))))
        return self.pool(F.relu(y + self.sc(x)))

class HTRVT(nn.Module):
    """Our implementation after HTR-VT (Li et al., Pattern Recognition 2025), = htr-metrics-transfer train_htrvt.ipynb: a residual CNN
    turns the 64 px line into a feature sequence (height 64 -> 4, folded into channels; width / 4), span masking on that sequence during
    training, sinusoidal positions, a pre-norm transformer encoder, a linear CTC head. Trained from scratch."""
    def __init__(self, n_classes, cnn_width, dim, depth, heads, mlp_ratio, dropout, mask_ratio, mask_span, **_):
        super().__init__()
        w1, w2, w3 = cnn_width // 4, cnn_width // 2, cnn_width
        self.cnn = nn.Sequential(nn.Conv2d(1, w1, 3, padding=1, bias=False), nn.BatchNorm2d(w1), nn.ReLU(),
                                 ResBlock(w1, w1, (2, 2)), ResBlock(w1, w2, (2, 2)), ResBlock(w2, w2, (2, 1)),
                                 ResBlock(w2, w3, (2, 1)), ResBlock(w3, w3, (1, 1)))
        self.proj = nn.Linear(w3 * 4, dim); self.mask_ratio, self.mask_span = mask_ratio, mask_span
        self.drop = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(dim, heads, int(dim * mlp_ratio), dropout, activation="gelu", batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm, self.head = nn.LayerNorm(dim), nn.Linear(dim, n_classes)
        pe = torch.zeros(4096, dim); pos = torch.arange(4096).float()[:, None]
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div); self.register_buffer("pe", pe, persistent=False)
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.trunc_normal_(m.weight, std=0.02); nn.init.zeros_(m.bias) if m.bias is not None else None
    def span_mask(self, x):
        B, T, _ = x.shape; n = max(1, int(T * self.mask_ratio / self.mask_span))
        st = torch.randint(0, max(1, T - self.mask_span), (B, n), device=x.device); m = torch.zeros(B, T, dtype=torch.bool, device=x.device)
        for o in range(self.mask_span): m.scatter_(1, (st + o).clamp(max=T - 1), True)
        return x.masked_fill(m[..., None], 0.0)
    def forward(self, px, lens=None):   # lens = CTC steps per line; padded steps are masked out of attention
        f = self.cnn(px); B, Cc, Hh, W = f.shape
        x = self.proj(f.permute(0, 3, 1, 2).reshape(B, W, Cc * Hh))
        if self.training and self.mask_ratio > 0: x = self.span_mask(x)
        pad = None if lens is None else torch.arange(W, device=x.device)[None, :] >= lens.to(x.device)[:, None]
        x = self.enc(self.drop(x + self.pe[:W]), src_key_padding_mask=pad)
        return F.log_softmax(self.head(self.norm(x)).float(), dim=-1)

model = HTRVT(charset.n_classes, **MODEL_CFG).to(device)
print(f"HTR-VT (ours) {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters, random init | max CTC steps per line {MAX_W // STRIDE}")

### 6. Training (validation = HHD writers, greedy CER)

In [ ]:
_N = str.maketrans({"״": '"', "׳": "'"})
VARIANTS = {"raw": lambda t: t,
            "norm": lambda t: re.sub(r"\s+", " ", t.translate(_N)).strip(),
            "let": lambda t: re.sub(r" +", " ", re.sub(r"[^א-ת ]", " ", t)).strip()}
def scores(R, H):
    o = {}
    for v, f in VARIANTS.items():
        rr = [f(x) for x in R]; hh = [f(x) for x in H]; k = [i for i, x in enumerate(rr) if x]
        rr = [rr[i] for i in k]; hh = [hh[i] for i in k]
        o[f"cer_{v}"] = jiwer.cer(rr, hh); o[f"wer_{v}"] = jiwer.wer(rr, hh)
    return o

@torch.no_grad()
def predict(split, beam=False):
    model.eval(); H, R = [], []
    for px, lens, _, texts in loader(split, False):
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=device == "cuda"):
            lp = model(px.to(device), lens)
        H += (beam_decode(lp.cpu(), lens, charset, beam_width=12, lm=char_lm, lm_weight=0.4) if beam
              else greedy_decode(lp.cpu(), lens, charset))
        R += texts
    return R, H

def ctc_step(px, lens, labels):
    keep = [i for i, l in enumerate(labels) if 0 < len(l) <= int(lens[i])]      # CTC-feasible lines
    if not keep: return None
    px, lens = px[keep].to(device), lens[keep]; labels = [labels[i] for i in keep]
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=device == "cuda"):
        lp = model(px, lens)                                                   # (B, T, C) float32
    return F.ctc_loss(lp.transpose(0, 1), torch.tensor(sum(labels, [])), lens, torch.tensor([len(l) for l in labels]),
                      blank=BLANK, zero_infinity=True)

def schedule(step, total, warm):
    if step < warm: return (step + 1) / warm
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))

LOG = os.path.join(OUT, "train_log.tsv")
open(LOG, "w").write("epoch\tstep\ttrain_loss\tval_cer_raw\tval_wer_raw\tval_blank_frac\tsec\n")
best = {"cer": 9.9, "state": None, "epoch": None}
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
total = EPOCHS * math.ceil(len(train_set(1)) / BATCH); warm = max(1, int(WARMUP * total)); step = 0; T0 = time.time()
for ep in range(1, EPOCHS + 1):
    augment.strength = AUG_MAX * min(1.0, 0.3 + 0.7 * (ep - 1) / max(1, AUG_RAMP - 1))
    model.train(); t0 = time.time(); losses = []
    for px, lens, labels, _ in loader(train_set(ep), True):
        for g in opt.param_groups: g["lr"] = LR * schedule(step, total, warm)
        loss = ctc_step(px, lens, labels); step += 1
        if loss is None: continue
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); losses.append(float(loss))
    line = f"{ep}\t{step}\t{np.mean(losses):.4f}"
    if ep % EVAL_EVERY == 0 or ep == EPOCHS:
        R, H = predict(val); s = scores(R, H); blank = float(np.mean([len(h) == 0 for h in H]))
        line += f"\t{s['cer_raw']:.4f}\t{s['wer_raw']:.4f}\t{blank:.3f}"
        print(f"[ep{ep}] loss {np.mean(losses):.3f} | val CER {s['cer_raw']*100:.2f} WER {s['wer_raw']*100:.2f} | empty {blank*100:.0f}% "
              f"| aug {augment.strength:.2f} | {time.time()-t0:.0f}s | total {(time.time()-T0)/3600:.2f} h", flush=True)
        if s["cer_raw"] < best["cer"]:
            best.update(cer=s["cer_raw"], epoch=ep, state={k: v.detach().to("cpu").clone() for k, v in model.state_dict().items()})
    else:
        line += "\t\t\t"
    open(LOG, "a").write(line + f"\t{time.time()-t0:.0f}\n")
TRAIN_HOURS = (time.time() - T0) / 3600
model.load_state_dict(best["state"]); print("best checkpoint: epoch", best["epoch"], "| val CER", round(best["cer"] * 100, 2), f"| {TRAIN_HOURS:.2f} h")

### 7. Final checks — HHD validation, held-out synthetic lines (digits / punctuation / number order)

In [ ]:
results = {}
for name, beam in (("greedy", False), ("beam12_charLM", True)):
    R, H = predict(val, beam=beam); results[name] = {"val": scores(R, H)}
    Rs, Hs = predict(syn_val, beam=beam); results[name]["syn_holdout"] = scores(Rs, Hs)
    v, w = results[name]["val"], results[name]["syn_holdout"]
    print(f"{name:14s} HHD val CER {v['cer_raw']*100:6.2f} WER {v['wer_raw']*100:6.2f} | "
          f"synthetic held-out CER {w['cer_raw']*100:6.2f} WER {w['wer_raw']*100:6.2f}")
same = rev = other = 0          # numbers: logical order learned? (labels are logical; after the RTL mirror the digits arrive reversed)
for r_, h_ in zip(Rs, Hs):
    for run in re.findall(r"\d{2,}", r_):
        if run in h_: same += 1
        elif run[::-1] in h_: rev += 1
        else: other += 1
tot = max(1, same + rev + other)
results["number_order_heldout"] = {"correct": same, "reversed": rev, "other": other}
print(f"multi-digit numbers in held-out synthetic: {tot} | correct order {100*same/tot:.2f}% | reversed {100*rev/tot:.2f}% | other {100*other/tot:.2f}%")
for i in range(4): print("\nREF ", Rs[i]); print("OUT ", Hs[i])

### 8. Save to the public HF repo

In [ ]:
from safetensors.torch import save_file
save_file({k: v.contiguous() for k, v in model.state_dict().items()}, os.path.join(OUT, "model.safetensors"))
charset.save(os.path.join(OUT, "charset.json"))
json.dump({"arch": "HTR-VT (ours): residual CNN + span masking + 6-layer transformer + linear CTC head, from scratch (exp40)",
           "model_cfg": MODEL_CFG, "stride": STRIDE, "row_h": ROW_H, "max_w": MAX_W, "n_classes": charset.n_classes,
           "preprocess": "gray, 64 px height, mirrored (RTL -> LTR), squeezed to <= max_w, [-1, 1], white padding",
           "sources": {"hhd": DATA_ID + " train+test", "sce": SCE_ID + " train+test", "pinkas": PINKAS_ID,
                       "synthetic": HHF_ID + f" (minus {SYN_HOLDOUT} held out)"},
           "mix": {"synthetic_frac": SYN_FRAC}, "train_lines": dict(src_lines), "density_target": DENSITY_TARGET,
           "train": {"lr": LR, "wd": WD, "warmup": WARMUP, "batch": BATCH, "epochs": EPOCHS, "eval_every": EVAL_EVERY,
                     "schedule": "linear warm-up + cosine decay to 1% of the peak", "augment": f"Mishkefet Augment {AUG_MAX}, ramp {AUG_RAMP}",
                     "seed": SEED, "hours": TRAIN_HOURS},
           "best_epoch": best["epoch"], "val_cer_raw": best["cer"], "compare_with": "cyttic/exp35-trocr-large-en-ctc-seqhead-allreal-synpunct",
           "note": "trained on HHD train+test, SCE, Pinkas: evaluate on the ivrit.ai benchmark only",
           "results": results}, open(os.path.join(OUT, "results.json"), "w"), indent=1)
open(os.path.join(OUT, "README.md"), "w").write(
    "---\nlicense: mit\nlanguage: he\ntags: [ocr, htr, hebrew, ctc]\n---\n# exp40 — HTR-VT (ours) on exp35's data mix\n\n"
    "Residual CNN + span masking + 6-layer transformer (dim 384) + linear CTC head, trained from scratch on HHD + SCE + Pinkas + 20% "
    "synthetic digits/punctuation lines. Input: grayscale line at 64 px height, mirrored (RTL to LTR). Charset: Mishkefet-v1. "
    f"HHD validation CER (greedy): {best['cer']*100:.2f}. Details in results.json.\n")
if PUSH and not SMOKE:
    api = HfApi(); api.create_repo(OUT_REPO, private=False, exist_ok=True)
    for k in range(5):
        try: api.upload_folder(repo_id=OUT_REPO, folder_path=OUT, commit_message="exp40 htrvt"); break
        except Exception as e: print("upload failed, retrying:", e); time.sleep(30)
    print("pushed -> https://huggingface.co/" + OUT_REPO)
else:
    print("not pushed (SMOKE or PUSH=False); files in", OUT)